In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

OUT = Path("paper_figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

METHOD_COLORS = {
    "Single Holdout": "#444444",
    "Expanding Window": "#0072B2",
    "Sliding Window": "#D55E00",
}


def save_figure(fig, filename):
    # These are newly rendered figures, not new experimental results.
    fig.savefig(
        OUT / f"{filename}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


# ============================================================
# FIGURE 2: Actual data boundaries
# Sources: Paper Tables 3-5 and backtest reports.
# Bar widths represent calendar time, not observation counts.
# ============================================================

train_ends = [
    "2013-12-31",
    "2015-12-31",
    "2017-12-29",
    "2019-12-31",
    "2021-12-31",
]

sliding_starts = [
    "2001-06-15",
    "2002-01-02",
    "2004-01-02",
    "2006-01-03",
    "2008-01-02",
]

validation = [
    ("2014-01-02", "2015-12-31"),
    ("2016-01-04", "2017-12-29"),
    ("2018-01-02", "2019-12-31"),
    ("2020-01-02", "2021-12-31"),
    ("2022-01-03", "2023-12-29"),
]

tests = [
    ("2016-01-04", "2017-12-29"),
    ("2018-01-02", "2019-12-31"),
    ("2020-01-02", "2021-12-31"),
    ("2022-01-03", "2023-12-29"),
    ("2024-01-02", "2026-07-28"),
]

rows = [{
    "label": "Single Holdout",
    "Train": ("2001-06-15", "2019-12-31"),
    "Validation": ("2020-01-02", "2023-12-29"),
    "Test": ("2024-01-02", "2026-07-28"),
}]

for i in range(5):
    rows.append({
        "label": f"Expanding F{i + 1}",
        "Train": ("2001-06-15", train_ends[i]),
        "Validation": validation[i],
        "Test": tests[i],
    })

for i in range(5):
    rows.append({
        "label": f"Sliding F{i + 1}",
        "Train": (sliding_starts[i], train_ends[i]),
        "Validation": validation[i],
        "Test": tests[i],
    })

split_colors = {
    "Train": "#0072B2",
    "Validation": "#E69F00",
    "Test": "#009E73",
}

fig, ax = plt.subplots(figsize=(12, 7))

# Light background identifies the shared historical test window.
ax.axvspan(
    pd.Timestamp("2024-01-02"),
    pd.Timestamp("2026-07-29"),
    color="#BBBBBB",
    alpha=0.18,
    zorder=0,
)

split_records = []

for y, row in enumerate(rows):
    for split in ["Train", "Validation", "Test"]:
        start, end = row[split]
        left = mdates.date2num(pd.Timestamp(start))
        # +1 day makes the stated end date inclusive visually.
        right = mdates.date2num(
            pd.Timestamp(end) + pd.Timedelta(days=1)
        )

        ax.barh(
            y,
            right - left,
            left=left,
            height=0.62,
            color=split_colors[split],
        )

        split_records.append({
            "experiment": row["label"],
            "split": split,
            "start": start,
            "end": end,
        })

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([row["label"] for row in rows])
ax.invert_yaxis()

ax.axhline(0.5, color="0.7", linewidth=0.8)
ax.axhline(5.5, color="0.7", linewidth=0.8)

ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.set_xlim(
    pd.Timestamp("2001-01-01"),
    pd.Timestamp("2027-01-01"),
)
ax.set_xlabel("Calendar year")
ax.set_title("Temporal data splits: actual available data")

legend_items = [
    Patch(color=split_colors[name], label=name)
    for name in ["Train", "Validation", "Test"]
]
legend_items.append(
    Patch(color="#BBBBBB", alpha=0.3, label="Common test window")
)
ax.legend(handles=legend_items, loc="lower left", ncol=4)

fig.text(
    0.01, 0.01,
    "Source: reported data boundaries. "
    "Bar lengths are calendar spans, not sample counts.",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.04, 1, 1))
save_figure(fig, "figure_2_temporal_splits")

pd.DataFrame(split_records).to_csv(
    OUT / "figure_2_source_dates.csv",
    index=False,
)


# ============================================================
# FIGURE 4: Reported fold metrics
# Source: Paper Table 9, checked against backtest reports.
# These are original reported results, not corrected reruns.
# ============================================================

metrics = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "expanding_sharpe": [2.09, 1.01, 0.38, -0.09, 2.01],
    "sliding_sharpe": [2.09, 1.06, 0.58, 0.18, 2.44],
    "expanding_maxdd_pct": [-5.56, -10.80, -13.76, -31.60, -9.77],
    "sliding_maxdd_pct": [-5.56, -10.59, -13.67, -26.83, -9.11],
})

x = np.arange(len(metrics))
width = 0.36

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for ax, exp_col, sl_col, ylabel in [
    (
        axes[0],
        "expanding_sharpe",
        "sliding_sharpe",
        "Reported Sharpe",
    ),
    (
        axes[1],
        "expanding_maxdd_pct",
        "sliding_maxdd_pct",
        "Maximum drawdown (%)",
    ),
]:
    ax.bar(
        x - width / 2,
        metrics[exp_col],
        width,
        color=METHOD_COLORS["Expanding Window"],
        label="Expanding Window",
    )
    ax.bar(
        x + width / 2,
        metrics[sl_col],
        width,
        color=METHOD_COLORS["Sliding Window"],
        label="Sliding Window",
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[0].legend()
axes[1].set_ylim(-36, 0)
axes[1].set_xticks(x)
axes[1].set_xticklabels([
    "F1\n2016-2017",
    "F2\n2018-2019",
    "F3\n2020-2021",
    "F4\n2022-2023",
    "F5\n2024-2026*",
])

fig.suptitle("Fold-level metrics from the original reported run")
fig.text(
    0.01, 0.01,
    "*Fold 5 data end: 2026-07-28. "
    "Unresolved implementation issues; no multi-seed uncertainty shown.",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.05, 1, 0.96))
save_figure(fig, "figure_4_reported_fold_metrics")

metrics.to_csv(
    OUT / "figure_4_source_metrics.csv",
    index=False,
)


# ============================================================
# FIGURE 5: Forward account snapshots
# Sources: Paper Tables 12,16 and the three inference reports.
# Use reported timestamps without assigning an unknown timezone.
# ============================================================

snapshot_data = {
    "Single Holdout": {
        "timestamp": [
            "2026-10-05 15:36:35",
            "2026-10-06 16:32:01",
            "2026-10-07 16:15:12",
            "2026-10-08 14:13:02",
            "2026-10-09 18:26:10",
        ],
        "net_liq": [
            999992.29,
            1008249.25,
            1000567.13,
            1002132.95,
            1008544.74,
        ],
    },
    "Expanding Window": {
        "timestamp": [
            "2026-10-05 15:36:30",
            "2026-10-06 16:24:33",
            "2026-10-07 16:15:09",
            "2026-10-08 14:12:40",
            "2026-10-09 18:26:10",
        ],
        "net_liq": [
            999983.04,
            1008575.80,
            1000457.19,
            1001929.33,
            1008510.25,
        ],
    },
    "Sliding Window": {
        "timestamp": [
            "2026-10-05 15:36:24",
            "2026-10-06 19:59:50",
            "2026-10-07 16:15:17",
            "2026-10-08 14:12:46",
            "2026-10-09 18:25:51",
        ],
        "net_liq": [
            1000000.00,
            1007253.94,
            1000519.00,
            1001961.33,
            1008458.66,
        ],
    },
}

markers = {
    "Single Holdout": "o",
    "Expanding Window": "s",
    "Sliding Window": "^",
}

fig, ax = plt.subplots(figsize=(10, 5.5))
snapshot_records = []

for method, values in snapshot_data.items():
    timestamps = pd.to_datetime(values["timestamp"])
    net_liq = np.asarray(values["net_liq"], dtype=float)
    index_value = 100.0 * net_liq / net_liq[0]

    ax.plot(
        timestamps,
        index_value,
        marker=markers[method],
        linestyle="--",
        linewidth=1.2,
        markersize=6,
        color=METHOD_COLORS[method],
        label=method,
    )

    for timestamp, value, index_point in zip(
        values["timestamp"], net_liq, index_value
    ):
        snapshot_records.append({
            "method": method,
            "reported_timestamp": timestamp,
            "timezone": "unspecified_in_report",
            "net_liq": float(value),
            "first_snapshot_100": float(index_point),
        })

ax.axhline(100, color="0.5", linewidth=0.8)
ax.set_ylabel("Account value index: first snapshot = 100")
ax.set_xlabel("Reported date and time; timezone unspecified")
ax.set_title("Forward account snapshots, 5-9 October 2026")
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.grid(alpha=0.2)
ax.legend()

fig.text(
    0.01, 0.01,
    "Five snapshots per method; timestamps are not synchronized. "
    "Dashed lines only connect observed points.",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_figure(fig, "figure_5_forward_snapshots")

pd.DataFrame(snapshot_records).to_csv(
    OUT / "figure_5_source_snapshots.csv",
    index=False,
)

print("Created figures and source tables in:", OUT.resolve())

Created figures and source tables in: D:\Trading Algorithm Project (Github)\Trading-Algorithm-Seminar-\Work15\paper_figures
